# VinRobotics VR M3.1 12DOF — Velocity Training (Kaggle, Run All)

Follows the repo README workflow: `Train -> Play`. Trains a **velocity-tracking walking policy** for task **`VR-M3-1-12DOF-Flat`** — **no motion/mocap data needed**, the robot learns to follow velocity commands from scratch.

**Output**: checkpoints at `logs/rsl_rl/vr_m3_1_12dof_velocity/<date_time>/model_<iteration>.pt` (+ `policy.onnx` auto-exported at each save), all bundled into `/kaggle/working/results_logs.zip`.

## Setup
- Settings → **Accelerator: GPU** (T4 best; P100 auto-applies a sparse-jacobian + CG-solver workaround), **Internet: ON**.
- **Run All**. To resume across sessions: download `results_logs.zip` from the Output tab, add it as an Input dataset next session, Run All again.

In [ ]:
# ================== CONFIG ==================
TASK = 'VR-M3-1-12DOF-Flat'    # or 'VR-M3-1-12DOF-Rough' / 'VR-M3-1-Flat' / 'VR-M3-1-Rough'
NUM_ENVS = 1024                # README uses 4096 (big GPU); 1024 is safe on Kaggle T4/P100 16GB
MAX_ITERS = 3000               # ~12h GPU/session; velocity task default is 20001 for a full run
GIT_URL = 'https://github.com/huytrao/vinrobotics_mjlab.git'
USE_WANDB = False              # True needs Kaggle secret WANDB_API_KEY
SAVE_INTERVAL = 500            # write model_<i>.pt every N iterations

PROJECT_DIR = '/kaggle/working/vinrobotics_mjlab'
print(f'Task={TASK}, envs={NUM_ENVS}, iters={MAX_ITERS}')

## 1. GPU check (auto P100/Pascal workaround)

In [ ]:
!nvidia-smi

import subprocess

# warp's libmathdx tile ops (tile_cholesky / tile_matmul) fail to compile on
# Pascal (sm_60: P100). Two mujoco_warp paths hit them: the dense-jacobian
# smooth solve, and the NEWTON solver's Hessian factorization. Sparse
# jacobian + CG solver avoids both.
PASCAL_FLAGS = ''
try:
    out = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,compute_cap', '--format=csv,noheader'],
        capture_output=True, text=True, check=True).stdout.strip()
    print('GPU(s):', out)
    for line in out.splitlines():
        gpu_name, cc = [x.strip() for x in line.split(',')]
        if int(cc.split('.')[0]) < 7:
            PASCAL_FLAGS = ('--env.sim.mujoco.jacobian=sparse '
                            '--env.sim.mujoco.solver=cg '
                            '--env.sim.mujoco.iterations=50')
            print(f'[WARN] {gpu_name} is Pascal (cc {cc}) -> applying workaround flags. '
                  'If it still crashes with an LTO error, switch Accelerator to T4 x2.')
except (FileNotFoundError, subprocess.CalledProcessError):
    print('nvidia-smi unavailable — no GPU attached? Check Settings -> Accelerator.')
print('PASCAL_FLAGS =', repr(PASCAL_FLAGS))

## 2. Get source (fresh-clone swap, safe against stale /kaggle/working)

In [ ]:
import os, shutil, stat, subprocess

def force_rmtree(path):
    def onerr(func, p, exc_info):
        try:
            os.chmod(p, stat.S_IWRITE)
            func(p)
        except OSError:
            pass
    if os.path.exists(path):
        shutil.rmtree(path, onerror=onerr)

if not os.path.exists(os.path.join(PROJECT_DIR, 'setup.py')):
    os.chdir('/kaggle/working')
    fresh = '/kaggle/working/_fresh_clone'
    force_rmtree(fresh)
    subprocess.run(['git', 'clone', GIT_URL, fresh], check=True)
    force_rmtree(PROJECT_DIR)
    if os.path.exists(PROJECT_DIR):
        shutil.copytree(fresh, PROJECT_DIR, dirs_exist_ok=True,
                        ignore=shutil.ignore_patterns('.git'))
        force_rmtree(fresh)
    else:
        os.rename(fresh, PROJECT_DIR)
else:
    print('Source already present, skipping clone.')

os.chdir(PROJECT_DIR)
print('OK — project at', PROJECT_DIR)

## 3. Restore checkpoints from a previous session (if `results_logs.zip` is an Input)

In [ ]:
import glob, zipfile

RESUME = False
prev = glob.glob('/kaggle/input/**/results_logs.zip', recursive=True)
if prev:
    print('Restoring logs from:', prev[0])
    with zipfile.ZipFile(prev[0]) as z:
        z.extractall(PROJECT_DIR)
    ckpts = glob.glob('logs/rsl_rl/**/model_*.pt', recursive=True)
    if ckpts:
        RESUME = True
        print(f'Restored {len(ckpts)} checkpoints -> training will RESUME.')
else:
    print('No previous results -> training from scratch.')

## 4. Install dependencies

In [ ]:
import subprocess, sys, os

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--upgrade', 'kaggle'], check=False)

!pip install -q mjlab==1.4.0 mujoco==3.8.1 mujoco-warp==3.8.1 warp-lang==1.13.0 prettytable
!pip install -q -e . --no-deps

os.environ['MUJOCO_GL'] = 'egl'
os.environ['PYOPENGL_PLATFORM'] = 'egl'

if USE_WANDB:
    from kaggle_secrets import UserSecretsClient
    os.environ['WANDB_API_KEY'] = UserSecretsClient().get_secret('WANDB_API_KEY')
else:
    os.environ['WANDB_MODE'] = 'offline'   # velocity rl_cfg defaults to logger='wandb'

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch==2.5.1', 'torchvision', 'torchaudio',
                '--index-url', 'https://download.pytorch.org/whl/cu121'], check=False)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'mlflow'], check=False)

import torch
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available(),
      '| device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu')

## 5. List tasks

In [ ]:
!python scripts/list_envs.py

## 6. Train (README command, no data needed)
Checkpoints `model_<iteration>.pt` land in `logs/rsl_rl/vr_m3_1_12dof_velocity/<date_time>/`; the velocity runner also writes `policy.onnx` next to them at every save.

In [ ]:
resume_flag = '--agent.resume True' if RESUME else ''
!python scripts/train.py {TASK} \
    --env.scene.num-envs={NUM_ENVS} \
    --agent.max-iterations={MAX_ITERS} \
    --agent.save-interval={SAVE_INTERVAL} \
    --gpu-ids '[0]' {PASCAL_FLAGS} {resume_flag}

## 7. Your .pt files

In [ ]:
import glob, os

ckpts = sorted(glob.glob('logs/rsl_rl/**/model_*.pt', recursive=True), key=os.path.getmtime)
if not ckpts:
    raise SystemExit('No checkpoints found — check the training cell output above for errors.')
for c in ckpts[-10:]:
    print(f'{c}  ({os.path.getsize(c)/1e6:.1f} MB)')
print('\nLatest .pt:', ckpts[-1])
onnx = sorted(glob.glob('logs/rsl_rl/**/*.onnx', recursive=True), key=os.path.getmtime)
if onnx:
    print('Latest .onnx:', onnx[-1])

## 8. Package results — download `results_logs.zip` from the Output tab

In [ ]:
!rm -f /kaggle/working/results_logs.zip
!zip -qr /kaggle/working/results_logs.zip logs -x '*wandb*'
!ls -lh /kaggle/working/results_logs.zip
print('\nDone! The zip contains all model_<i>.pt checkpoints + policy.onnx + configs.')